# Improved training, evaluation, and Grad-CAM

This is the interactive front end for the reusable implementation in `src/` and `scripts/`. It trains only on `train`, chooses the checkpoint from `valid`, then reports held-out `test` metrics and produces a localization heatmap for a random image.

> Grad-CAM highlights regions that influenced the model prediction. It is not a clinical segmentation or diagnosis.

In [ ]:
from pathlib import Path
import sys

# Run this notebook from the repository root.
DATA_DIR = Path('data/images')  # contains train/, valid/, and test/
CHECKPOINT = Path('models/skin_lesion_mobilenet.pth')
TEST_DIR = DATA_DIR / 'test'
IMAGE_SIZE = 224
ARCHITECTURE = 'mobilenet_v2'  # or 'resnet18'

assert (DATA_DIR / 'train').is_dir(), 'Create data/images/train/<class-name>/ first'
assert (DATA_DIR / 'valid').is_dir(), 'Create data/images/valid/<class-name>/ first'
assert (DATA_DIR / 'test').is_dir(), 'Create data/images/test/<class-name>/ first'

## Train

This uses ImageNet normalization, medical-image-safe spatial/color augmentation, weighted sampling for imbalanced classes, label smoothing, macro-F1 checkpointing, learning-rate reduction, and early stopping.

In [ ]:
!{sys.executable} scripts/train.py --data-dir {DATA_DIR} --architecture {ARCHITECTURE} --image-size {IMAGE_SIZE} --epochs 25 --output {CHECKPOINT}

## Held-out test metrics

Do not tune hyperparameters using this test set; use it once after selecting the validation checkpoint.

In [ ]:
!{sys.executable} scripts/evaluate.py --checkpoint {CHECKPOINT} --test-dir {TEST_DIR}

## Random-image Grad-CAM

A different image is chosen by changing `--seed`. Pass `--image path/to/image.jpg` to inspect a particular image instead.

In [ ]:
HEATMAP = Path('outputs/random_gradcam.png')
!{sys.executable} scripts/infer_random.py --checkpoint {CHECKPOINT} --image-dir {TEST_DIR} --output {HEATMAP} --seed 42

In [ ]:
from IPython.display import Image, display
display(Image(filename=HEATMAP))